Support Ticket Triage with Guardrails

In [4]:
import sys
import pandas as pd
import numpy as np
import re
import json
import random
import time

from pathlib import Path
from sklearn.metrics import accuracy_score, confusion_matrix

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import mock_llm

TICKETS_PATH = Path("../data/raw/tickets.csv")

SEGMENTS_PATH = Path("../outputs/customer_segments.csv")

OUTPUT_DIR = Path("../outputs")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
MAX_INPUT_TOKENS = 400

TOKEN_BUDGET = 6000

FINAL_TOKEN_BUDGET = 100000

BASE_DELAY = 0.05

MAX_RETRIES = 3

ALLOWED_CATEGORIES = ["billing","delivery","technical","fraud"]


In [6]:
tickets = pd.read_csv(TICKETS_PATH)

segments = pd.read_csv(SEGMENTS_PATH)

print("Tickets shape:", tickets.shape)

print("\nTicket columns:")
print(tickets.columns.tolist())

print("\nSegments shape:", segments.shape)

print("\nSegment columns:")
print(segments.columns.tolist())

Tickets shape: (164, 5)

Ticket columns:
['ticket_id', 'customer_id', 'order_id', 'text', 'true_category']

Segments shape: (1175, 7)

Segment columns:
['customer_id', 'recency', 'frequency', 'monetary', 'segment_id', 'segment_name', 'is_premium']


In [7]:
required_ticket_columns = {"ticket_id","customer_id","order_id","text","true_category"}

required_segment_columns = {"customer_id","segment_name","is_premium"}

assert required_ticket_columns.issubset(tickets.columns)

assert required_segment_columns.issubset(segments.columns)

In [8]:
tickets["customer_id"] = (
    tickets["customer_id"]
    .astype("string")
    .str.strip()
)

segments["customer_id"] = (
    segments["customer_id"]
    .astype("string")
    .str.strip()
)

In [9]:
def check_input_guardrails(text):

    if pd.isna(text) or str(text).strip() == "":
        return "skipped_empty", 0

    text = str(text)

    estimated_tokens = mock_llm.est_tokens(text)

    if estimated_tokens > MAX_INPUT_TOKENS:
        return "flagged_long", estimated_tokens

    return "eligible", estimated_tokens

In [10]:
tickets["text"] = tickets["text"].fillna("")

tickets["input_status"] = None

tickets["estimated_tokens"] = 0

for index, row in tickets.iterrows():

    status, token_count = check_input_guardrails(
        row["text"]
    )

    tickets.loc[index, "input_status"] = status

    tickets.loc[index, "estimated_tokens"] = token_count

In [12]:
print(tickets["input_status"].value_counts())

assert (tickets["input_status"] == "skipped_empty").sum() == 4

assert (tickets["input_status"] == "flagged_long").sum() == 4

assert (tickets["input_status"] == "eligible").sum() == 156

input_status
eligible         156
skipped_empty      4
flagged_long       4
Name: count, dtype: int64


In [13]:
EMAIL_PATTERN = r"[\w.+-]+@[\w-]+\.[\w.]+"

PHONE_PATTERN = r"\b[6-9]\d{9}\b"

ORDER_PATTERN = r"ORD-\d{6}"

In [14]:
def redact_pii(text):

    email_count = 0
    phone_count = 0
    order_count = 0

    text = str(text)

    text, email_count = re.subn(EMAIL_PATTERN,"[EMAIL]",text)

    text, phone_count = re.subn(PHONE_PATTERN,"[PHONE]",text)

    text, order_count = re.subn(ORDER_PATTERN,"[ORDER_ID]",text)

    return (
        text,
        email_count,
        phone_count,
        order_count
    )

In [15]:
tickets["redacted_text"] = ""

total_emails = 0
total_phones = 0
total_orders = 0

for index, row in tickets.iterrows():

    (
        redacted_text,
        email_count,
        phone_count,
        order_count
    ) = redact_pii(row["text"])

    tickets.loc[index, "redacted_text"] = redacted_text

    total_emails += email_count

    total_phones += phone_count

    total_orders += order_count

In [16]:
print("Emails redacted:", total_emails)

print("Phone numbers redacted:", total_phones)

print("Order IDs redacted:", total_orders)

print(
    "Total replacements:",
    total_emails + total_phones + total_orders
)

Emails redacted: 55
Phone numbers redacted: 91
Order IDs redacted: 132
Total replacements: 278


In [17]:
def contains_pii(text):

    patterns = [EMAIL_PATTERN,PHONE_PATTERN,ORDER_PATTERN]

    for pattern in patterns:

        if re.search(pattern, str(text)):
            return True

    return False

assert not tickets["redacted_text"].apply(contains_pii).any()

print("PII validation successful.")

PII validation successful.


In [18]:
zero_shot_prompt = """
CLASSIFY the customer support ticket into exactly one of these categories:

1. billing
2. delivery
3. technical
4. fraud

Return only valid JSON in the following format:

{"category": "...", "reason": "..."}

The category must be one of the four categories listed above.

Do not include markdown, explanations outside JSON, or additional text.
"""

In [19]:
few_shot_prompt = """
CLASSIFY the customer support ticket into exactly one of these categories:

1. billing
2. delivery
3. technical
4. fraud

EXAMPLES:

Ticket: I was charged twice for my order.
Output: {"category": "billing", "reason": "Duplicate charge"}

Ticket: My parcel has not arrived even though delivery was promised.
Output: {"category": "delivery", "reason": "Delayed delivery"}

Ticket: Someone accessed my account without permission.
Output: {"category": "fraud", "reason": "Unauthorized account access"}

Ticket: My application crashes whenever I open it.
Output: {"category": "technical", "reason": "Application crash"}

Return only valid JSON in the following format:

{"category": "...", "reason": "..."}

The category must be one of the four categories listed above.

Do not include markdown, explanations outside JSON, or additional text.
"""

In [20]:
assert "CLASSIFY" in zero_shot_prompt

assert "CLASSIFY" in few_shot_prompt

assert "EXAMPLES:" not in zero_shot_prompt

assert "EXAMPLES:" in few_shot_prompt

for category in ALLOWED_CATEGORIES:

    assert category in zero_shot_prompt

    assert category in few_shot_prompt

print("Prompt validation successful.")

Prompt validation successful.


In [27]:
def call_llm(system_prompt, redacted_text, ticket_id):

    assert not contains_pii(redacted_text), (
        f"PII detected in ticket {ticket_id}"
    )

    messages = [

        {
            "role": "system",
            "content": system_prompt
        },

        {
            "role": "user",
            "content": redacted_text
        }

    ]

    assert messages[1]["content"] == redacted_text

    retry_count = 0

    while True:

        try:

            response = mock_llm.chat(messages)

            output_text = (
                response.choices[0]
                .message.content
            )

            tokens_used = (
                response.usage.total_tokens
            )

            return output_text, tokens_used, retry_count

        except mock_llm.TransientLLMError:

            if retry_count >= MAX_RETRIES:

                raise

            wait_time = (
                BASE_DELAY
                * (2 ** retry_count)
                * random.uniform(0.9, 1.15)
            )

            retry_count += 1

            print(
                f"Ticket {ticket_id} | "
                f"Retry {retry_count}/{MAX_RETRIES} | "
                f"Wait: {wait_time:.4f} seconds"
            )

            time.sleep(wait_time)

In [28]:
def validate_output(output_text):

    try:

        result = json.loads(output_text)

        category = result.get("category")

        if category in ALLOWED_CATEGORIES:

            return category, "classified_json"

        return None, "flagged_invalid_output"

    except (json.JSONDecodeError, AttributeError):

        words = re.findall(
            r"[a-z]+",
            output_text.lower()
        )

        matched_categories = [
            word
            for word in words
            if word in ALLOWED_CATEGORIES
        ]

        if len(matched_categories) == 1:

            return (
                matched_categories[0],
                "classified_keyword"
            )

        return None, "flagged_invalid_output"

In [29]:
def run_pipeline(
    data,
    system_prompt,
    token_budget=None
):

    mock_llm.reset()

    results = []

    running_tokens = 0

    total_retries = 0

    for _, row in data.iterrows():

        ticket_id = row["ticket_id"]

        customer_id = row["customer_id"]

        original_status = row["input_status"]

        redacted_text = row["redacted_text"]

        category = None

        tokens_used = 0

        retry_count = 0

        if original_status == "skipped_empty":

            status = "skipped_empty"

        elif original_status == "flagged_long":

            status = "flagged_long"

        else:

            if (
                token_budget is not None
                and running_tokens >= token_budget
            ):

                status = "budget_exhausted"

            else:

                output_text, tokens_used, retry_count = (
                    call_llm(
                        system_prompt,
                        redacted_text,
                        ticket_id
                    )
                )

                running_tokens += tokens_used

                total_retries += retry_count

                category, status = validate_output(
                    output_text
                )

        results.append({

            "ticket_id": ticket_id,

            "customer_id": customer_id,

            "status": status,

            "category": category,

            "tokens_used": tokens_used,

            "retry_count": retry_count,

            "true_category": row["true_category"]

        })

    results_df = pd.DataFrame(results)

    return results_df, running_tokens, total_retries

In [30]:
zero_shot_results, zero_shot_tokens, zero_shot_retries = (
    run_pipeline(
        tickets,
        zero_shot_prompt
    )
)

Ticket T0002 | Retry 1/3 | Wait: 0.0549 seconds
Ticket T0116 | Retry 1/3 | Wait: 0.0504 seconds
Ticket T0153 | Retry 1/3 | Wait: 0.0453 seconds
Ticket T0039 | Retry 1/3 | Wait: 0.0453 seconds
Ticket T0152 | Retry 1/3 | Wait: 0.0520 seconds
Ticket T0043 | Retry 1/3 | Wait: 0.0481 seconds
Ticket T0111 | Retry 1/3 | Wait: 0.0479 seconds
Ticket T0155 | Retry 1/3 | Wait: 0.0561 seconds
Ticket T0100 | Retry 1/3 | Wait: 0.0494 seconds
Ticket T0068 | Retry 1/3 | Wait: 0.0567 seconds
Ticket T0135 | Retry 1/3 | Wait: 0.0562 seconds
Ticket T0009 | Retry 1/3 | Wait: 0.0564 seconds
Ticket T0067 | Retry 1/3 | Wait: 0.0565 seconds
Ticket T0106 | Retry 1/3 | Wait: 0.0504 seconds
Ticket T0097 | Retry 1/3 | Wait: 0.0475 seconds
Ticket T0003 | Retry 1/3 | Wait: 0.0569 seconds
Ticket T0019 | Retry 1/3 | Wait: 0.0495 seconds
Ticket T0086 | Retry 1/3 | Wait: 0.0571 seconds
Ticket T0065 | Retry 1/3 | Wait: 0.0542 seconds


In [31]:
print(zero_shot_results.head())

  ticket_id customer_id           status category  tokens_used  retry_count  \
0     T0024       C0486  classified_json    fraud          100            0   
1     T0073       C0080  classified_json  billing           96            0   
2     T0095       C1087  classified_json  billing           98            0   
3     T0159       C1172    skipped_empty      NaN            0            0   
4     T0080       C0910  classified_json  billing          100            0   

  true_category  
0         fraud  
1       billing  
2     technical  
3          none  
4         fraud  


In [32]:
print(zero_shot_results["status"].value_counts())

status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
skipped_empty               4
flagged_long                4
Name: count, dtype: int64


In [33]:
def calculate_metrics(results, total_tokens):

    classified = results[
        results["status"].isin([
            "classified_json",
            "classified_keyword"
        ])
    ].copy()

    classified_count = len(classified)

    accuracy = accuracy_score(
        classified["true_category"],
        classified["category"]
    )

    tokens_per_ticket = (
        total_tokens / classified_count
        if classified_count > 0
        else 0
    )

    return {

        "classified_count": classified_count,

        "accuracy": accuracy,

        "total_tokens": total_tokens,

        "tokens_per_classified_ticket": tokens_per_ticket

    }

In [34]:
zero_metrics = calculate_metrics(
    zero_shot_results,
    zero_shot_tokens
)

print("Zero-shot metrics:")

for key, value in zero_metrics.items():

    print(key, ":", value)

Zero-shot metrics:
classified_count : 147
accuracy : 0.8571428571428571
total_tokens : 15016
tokens_per_classified_ticket : 102.14965986394557


In [35]:
zero_classified = zero_shot_results[
    zero_shot_results["status"].isin([
        "classified_json",
        "classified_keyword"
    ])
]

cm = confusion_matrix(
    zero_classified["true_category"],
    zero_classified["category"],
    labels=ALLOWED_CATEGORIES
)

cm_df = pd.DataFrame(
    cm,
    index=ALLOWED_CATEGORIES,
    columns=ALLOWED_CATEGORIES
)

print(cm_df)

           billing  delivery  technical  fraud
billing         38         0          0      0
delivery         0        36          0      0
technical       10         0         29      0
fraud           11         0          0     23


In [36]:
few_shot_results, few_shot_tokens, few_shot_retries = (
    run_pipeline(
        tickets,
        few_shot_prompt
    )
)

Ticket T0002 | Retry 1/3 | Wait: 0.0478 seconds
Ticket T0116 | Retry 1/3 | Wait: 0.0465 seconds
Ticket T0153 | Retry 1/3 | Wait: 0.0565 seconds
Ticket T0039 | Retry 1/3 | Wait: 0.0479 seconds
Ticket T0152 | Retry 1/3 | Wait: 0.0569 seconds
Ticket T0043 | Retry 1/3 | Wait: 0.0547 seconds
Ticket T0111 | Retry 1/3 | Wait: 0.0455 seconds
Ticket T0155 | Retry 1/3 | Wait: 0.0498 seconds
Ticket T0100 | Retry 1/3 | Wait: 0.0476 seconds
Ticket T0068 | Retry 1/3 | Wait: 0.0483 seconds
Ticket T0135 | Retry 1/3 | Wait: 0.0477 seconds
Ticket T0009 | Retry 1/3 | Wait: 0.0470 seconds
Ticket T0067 | Retry 1/3 | Wait: 0.0572 seconds
Ticket T0106 | Retry 1/3 | Wait: 0.0470 seconds
Ticket T0097 | Retry 1/3 | Wait: 0.0481 seconds
Ticket T0003 | Retry 1/3 | Wait: 0.0552 seconds
Ticket T0019 | Retry 1/3 | Wait: 0.0549 seconds
Ticket T0086 | Retry 1/3 | Wait: 0.0557 seconds
Ticket T0065 | Retry 1/3 | Wait: 0.0464 seconds


In [37]:
few_metrics = calculate_metrics(
    few_shot_results,
    few_shot_tokens
)

print("Few-shot metrics:")

for key, value in few_metrics.items():

    print(key, ":", value)

Few-shot metrics:
classified_count : 147
accuracy : 1.0
total_tokens : 27184
tokens_per_classified_ticket : 184.9251700680272


In [38]:
comparison = pd.DataFrame({

    "Metric": [
        "Classified tickets",
        "Accuracy",
        "Total tokens",
        "Tokens per classified ticket"
    ],

    "Zero-shot": [
        zero_metrics["classified_count"],
        zero_metrics["accuracy"],
        zero_metrics["total_tokens"],
        zero_metrics["tokens_per_classified_ticket"]
    ],

    "Few-shot": [
        few_metrics["classified_count"],
        few_metrics["accuracy"],
        few_metrics["total_tokens"],
        few_metrics["tokens_per_classified_ticket"]
    ]

})

print(comparison.to_string(index=False))

                      Metric    Zero-shot    Few-shot
          Classified tickets   147.000000   147.00000
                    Accuracy     0.857143     1.00000
                Total tokens 15016.000000 27184.00000
Tokens per classified ticket   102.149660   184.92517


In [40]:
budget_results, budget_tokens, budget_retries = (
    run_pipeline(
        tickets,
        zero_shot_prompt,
        token_budget=6000
    )
)

Ticket T0002 | Retry 1/3 | Wait: 0.0493 seconds
Ticket T0116 | Retry 1/3 | Wait: 0.0547 seconds
Ticket T0153 | Retry 1/3 | Wait: 0.0452 seconds
Ticket T0039 | Retry 1/3 | Wait: 0.0486 seconds
Ticket T0152 | Retry 1/3 | Wait: 0.0468 seconds
Ticket T0043 | Retry 1/3 | Wait: 0.0543 seconds
Ticket T0111 | Retry 1/3 | Wait: 0.0555 seconds


In [41]:
print(budget_results["status"].value_counts())

status
budget_exhausted          93
classified_json           56
skipped_empty              4
flagged_long               4
classified_keyword         4
flagged_invalid_output     3
Name: count, dtype: int64


In [42]:
exhausted_count = (
    budget_results["status"] == "budget_exhausted"
).sum()

print(
    "Budget exhausted tickets:",
    exhausted_count
)

Budget exhausted tickets: 93


In [43]:
if exhausted_count == 0:

    budget_results, budget_tokens, budget_retries = (
        run_pipeline(
            tickets,
            zero_shot_prompt,
            token_budget=5000
        )
    )

    exhausted_count = (
        budget_results["status"] == "budget_exhausted"
    ).sum()

    print("New budget:", 5000)

    print("Exhausted tickets:", exhausted_count)

In [44]:
budget_results.to_csv(OUTPUT_DIR / "triage_budget_demo.csv",index=False)

In [45]:
final_results, final_tokens, final_retries = (
    run_pipeline(
        tickets,
        few_shot_prompt,
        token_budget=100000
    )
)

Ticket T0002 | Retry 1/3 | Wait: 0.0544 seconds
Ticket T0116 | Retry 1/3 | Wait: 0.0531 seconds
Ticket T0153 | Retry 1/3 | Wait: 0.0565 seconds
Ticket T0039 | Retry 1/3 | Wait: 0.0473 seconds
Ticket T0152 | Retry 1/3 | Wait: 0.0492 seconds
Ticket T0043 | Retry 1/3 | Wait: 0.0489 seconds
Ticket T0111 | Retry 1/3 | Wait: 0.0485 seconds
Ticket T0155 | Retry 1/3 | Wait: 0.0540 seconds
Ticket T0100 | Retry 1/3 | Wait: 0.0469 seconds
Ticket T0068 | Retry 1/3 | Wait: 0.0527 seconds
Ticket T0135 | Retry 1/3 | Wait: 0.0547 seconds
Ticket T0009 | Retry 1/3 | Wait: 0.0476 seconds
Ticket T0067 | Retry 1/3 | Wait: 0.0503 seconds
Ticket T0106 | Retry 1/3 | Wait: 0.0514 seconds
Ticket T0097 | Retry 1/3 | Wait: 0.0479 seconds
Ticket T0003 | Retry 1/3 | Wait: 0.0496 seconds
Ticket T0019 | Retry 1/3 | Wait: 0.0474 seconds
Ticket T0086 | Retry 1/3 | Wait: 0.0533 seconds
Ticket T0065 | Retry 1/3 | Wait: 0.0559 seconds


In [46]:
print(final_results["status"].value_counts())

status
classified_json           135
classified_keyword         12
flagged_invalid_output      9
skipped_empty               4
flagged_long                4
Name: count, dtype: int64


In [47]:
assert len(final_results) == 164

assert final_retries == 19

assert (final_results["status"] == "classified_json").sum() == 135

assert (final_results["status"] == "classified_keyword").sum() == 12

assert (final_results["status"] == "flagged_invalid_output").sum() == 9

assert (final_results["status"] == "skipped_empty").sum() == 4

assert (final_results["status"] == "flagged_long").sum() == 4

print("Final classification checks passed.")

Final classification checks passed.


In [48]:
segment_info = segments[["customer_id","segment_name","is_premium"]].copy()

def convert_to_bool(value):

    if pd.isna(value):
        return False

    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    return str(value).strip().lower() in [
        "true",
        "1",
        "yes",
        "y"
    ]


segment_info["is_premium"] = (
    segment_info["is_premium"]
    .apply(convert_to_bool)
)

In [49]:
routed_results = final_results.merge(
    segment_info,
    on="customer_id",
    how="left"
)

In [50]:
routed_results["segment_name"] = (
    routed_results["segment_name"]
    .fillna("")
)

routed_results["is_premium"] = (
    routed_results["is_premium"]
    .fillna(False)
)

In [51]:
print("Rows before merge:", len(final_results))

print("Rows after merge:", len(routed_results))

Rows before merge: 164
Rows after merge: 164


In [52]:
def assign_routing(row):

    status = row["status"]

    category = row["category"]

    is_premium = row["is_premium"]

    if status in [
        "skipped_empty",
        "flagged_long",
        "flagged_invalid_output",
        "budget_exhausted"
    ]:

        return "human_review", "NORMAL"

    if category == "fraud":

        return "security_team", "HIGH"

    if is_premium:

        return "premium_desk", "HIGH"

    return "support_queue", "NORMAL"

In [53]:
routed_results[
    ["route", "priority"]
] = routed_results.apply(
    lambda row: pd.Series(
        assign_routing(row)
    ),
    axis=1
)

In [54]:
print(routed_results["route"].value_counts())

print("\nPriority counts:")

print(routed_results["priority"].value_counts())

route
support_queue    102
security_team     34
human_review      17
premium_desk      11
Name: count, dtype: int64

Priority counts:
priority
NORMAL    119
HIGH       45
Name: count, dtype: int64


In [55]:
print(
    routed_results[
        [
            "ticket_id",
            "customer_id",
            "status",
            "category",
            "segment_name",
            "is_premium",
            "route",
            "priority"
        ]
    ].head(15)
)

   ticket_id customer_id              status   category          segment_name  \
0      T0024       C0486     classified_json      fraud     At-Risk Customers   
1      T0073       C0080     classified_json    billing        Loyal Regulars   
2      T0095       C1087     classified_json  technical  Developing Customers   
3      T0159       C1172       skipped_empty        NaN        Loyal Regulars   
4      T0080       C0910     classified_json      fraud  Developing Customers   
5      T0124       C0139     classified_json      fraud        Loyal Regulars   
6      T0061       C0812     classified_json    billing        Loyal Regulars   
7      T0118       C0355     classified_json   delivery        Loyal Regulars   
8      T0037       C1156     classified_json    billing     Premium Customers   
9      T0002       C0881     classified_json   delivery  Developing Customers   
10     T0163       C0322        flagged_long        NaN        Loyal Regulars   
11     T0031       C0958    

In [56]:
final_output = routed_results[
    [
        "ticket_id",
        "customer_id",
        "status",
        "category",
        "segment_name",
        "priority",
        "route",
        "tokens_used"
    ]
].copy()

assert len(final_output) == 164

assert final_output.columns.tolist() == [
    "ticket_id",
    "customer_id",
    "status",
    "category",
    "segment_name",
    "priority",
    "route",
    "tokens_used"
]

assert (
    final_output["ticket_id"].tolist()
    == tickets["ticket_id"].tolist()
)

In [57]:
final_output.to_csv(OUTPUT_DIR / "triage_results.csv",index=False)

In [58]:
assert len(final_output) == 164

assert final_output.columns.tolist() == [
    "ticket_id",
    "customer_id",
    "status",
    "category",
    "segment_name",
    "priority",
    "route",
    "tokens_used"
]

assert (
    final_results.loc[
        final_results["status"] == "skipped_empty",
        "tokens_used"
    ].sum()
    == 0
)

assert (
    final_results.loc[
        final_results["status"] == "flagged_long",
        "tokens_used"
    ].sum()
    == 0
)

assert total_emails == 55

assert total_phones == 91

assert total_orders == 132

assert not tickets["redacted_text"].apply(
    contains_pii
).any()

assert final_retries == 19

assert (
    final_results["status"] == "classified_json"
).sum() == 135

assert (
    final_results["status"] == "classified_keyword"
).sum() == 12

assert (
    final_results["status"] == "flagged_invalid_output"
).sum() == 9


assert final_output["route"].notna().all()

assert (OUTPUT_DIR / "triage_results.csv").exists()


print("All Part C validation checks passed!")

All Part C validation checks passed!
